In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

from Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets
from Pro.DataPrep import Prep_Map, Output_Map
from College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map

In [ ]:
data_prep = Combined_Data_Prep(
    Prep_Map.base_prep_map, 
    Output_Map.base_output_map,
    C_Prep_Map.college_base_prep_map,
    C_Output_Map.college_output_map)

pither_io_list = data_prep.Generate_IO_Pitchers(pro_player_condition="WHERE lastMLBSeason<? AND signingYear<? AND isPitcher=?", pro_player_values=(2025,2015,1), pro_use_cutoff=True,
                                        col_player_condition="WHERE LastYear<=? AND isPitcher=?", col_player_values=(2015, 1), col_use_cutoff=True)


In [ ]:
train_dataset, test_dataset = Create_Test_Train_Datasets(pither_io_list, 0.25, 0, False)

In [ ]:
from Pro.Model.Player_Model import RNN_Model as Pro_Model, LayerArch
from College.Model.College_Model import RNN_Model as College_Model

import torch
from Constants import device

mlbstat_arch = LayerArch(layer_size=20, num_layers=2)

pro_model = Pro_Model(
    input_size=train_dataset.GetProInputSize(),
    mutators=torch.empty(0),
    data_prep=data_prep.pro_data_prep,
    is_hitter=False,
    mlbstat_arch=mlbstat_arch,
).to(device)
col_model = College_Model(
    input_size=train_dataset.GetColInputSize(),
    data_prep=data_prep.college_data_prep,
    is_hitter=False,
    output_hidden_size=pro_model.GetHiddenSize(),
    output_num_layers=pro_model.GetNumLayers(),
).to(device)

In [ ]:
from Combined.Model.Model_Train import TrainAndGraph

best_loss = TrainAndGraph(
    pro_network=pro_model,
    col_network=col_model,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    pro_model_name="../Models/test_pro_pit",
    col_model_name="../Models/test_col_pit",
    is_hitter=False
)

print(best_loss)